<a href="https://colab.research.google.com/github/k1-r1ll/pytorch/blob/main/14_pytorch_transform_imagefolder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
from PIL import Image

import torch
import torch.utils.data as data
import torchvision
import torchvision.transforms.v2 as tfs
from torchvision.datasets import ImageFolder

import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

In [2]:
# {
#   "class_0": 0,
#   "class_1": 1,
#   "class_2": 2,
#   "class_3": 3,
#   "class_4": 4,
#   "class_5": 5,
#   "class_6": 6,
#   "class_7": 7,
#   "class_8": 8,
#   "class_9": 9
# }

In [3]:
transform = tfs.ToPILImage()

mnist_train = torchvision.datasets.MNIST('mnist', download=True, train=True)
mnist_test = torchvision.datasets.MNIST('mnist', download=True, train=False)

dir_out = 'dataset'
file_format = 'format.json'

train_data = {'dir': 'train', 'data': mnist_train}
test_data = {'dir': 'test', 'data': mnist_test}

if not os.path.exists(dir_out):
    os.makedirs(dir_out, exist_ok=True)

for info in (train_data, test_data):
    os.makedirs(os.path.join(dir_out, info['dir']), exist_ok=True)

    for i in range(10):
        os.makedirs(os.path.join(dir_out, info['dir'], f"class_{i}"), exist_ok=True)

    print(f"Начинаю сохранять картинки для {info['dir']}")

    for index, (img, label) in enumerate(info['data']):

        filename = f"img_{index}.png"
        path_to_save = os.path.join(dir_out, info['dir'], f"class_{label}", filename)
        if not isinstance(img, Image.Image):
            img = transform(img)

        img.save(path_to_save)

    print(f"Картинки для {info['dir']} успешно сохранены")

100%|██████████| 9.91M/9.91M [00:00<00:00, 15.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 406kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.80MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.01MB/s]


Начинаю сохранять картинки для train
Картинки для train успешно сохранены
Начинаю сохранять картинки для test
Картинки для test успешно сохранены


In [5]:
transforms = tfs.Compose([tfs.ToImage(),
                          tfs.Grayscale(),
                          tfs.ToDtype(torch.float32, scale=True),
                          tfs.Lambda(lambda _img: _img.ravel())])
d_train = ImageFolder("dataset/train", transform=transforms)
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)

In [6]:
class DigitNN(nn.Module):
  def __init__(self, imp_dim, hid_dim, out_dim):
      super().__init__()
      self.l1 = nn.Linear(imp_dim, hid_dim)
      self.l2 = nn.Linear(hid_dim, out_dim)

  def forward(self, x):
    x = self.l1(x)
    x = nn.functional.relu(x)
    x = self.l2(x)
    return x

In [7]:
model = DigitNN(28 * 28, 32, 10)
optimizer = optim.Adam(params=model.parameters(), lr=0.01)
loss_func = nn.CrossEntropyLoss()
epochs = 2
model.train()

DigitNN(
  (l1): Linear(in_features=784, out_features=32, bias=True)
  (l2): Linear(in_features=32, out_features=10, bias=True)
)

In [8]:
for _e in range(epochs):
  loss_mean = 0
  lm_count = 0


  train_tqdm = tqdm(train_data, leave=True)

  for x_train, y_train in train_tqdm:
    predict = model(x_train)
    loss = loss_func(predict, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    lm_count += 1

    loss_mean = 1 / lm_count * loss.item() + (1 - 1 / lm_count) * loss_mean
    train_tqdm.set_description(f"Epoch {_e + 1}/{epochs}, loss_mean: {loss_mean}")


Epoch 2/2, loss_mean: 0.17861263623523171: 100%|██████████| 1875/1875 [00:38<00:00, 48.13it/s]


In [10]:
d_test = ImageFolder("dataset/test", transform=transforms)
test_data = data.DataLoader(d_test, batch_size=500, shuffle=False)
Q = 0

In [11]:
model.eval()

DigitNN(
  (l1): Linear(in_features=784, out_features=32, bias=True)
  (l2): Linear(in_features=32, out_features=10, bias=True)
)

In [12]:
for x_test, y_test in test_data:
  with torch.no_grad():
    p = model(x_test)
    p = torch.argmax(p, dim=1)
    Q += torch.sum(p == y_test).item()

In [13]:
Q /= len(d_test)
print(Q)

0.9505
